# Research Project: GAN-Based Image Deblurring for Robust CNN-Based Road Damage Classification
### End-to-End Google Colab Pipeline: Data Preparation, GAN Deblurring, Baseline & Restored Classification

**Core Research Question:** Does GAN-based deblurring improve lightweight CNN road-damage classification under image blur, and are any gains worth the added inference time and computational cost?

**Important Scope Decision:** This project implements **Patch-Level Road Damage Classification** (Normal, Longitudinal Crack, Transverse Crack, Alligator Crack, Pothole), isolating image degradation and restoration effects from object detection.

## Step 1: Environment Setup & Hardware Check

In [ ]:
# Check GPU accelerator
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
else:
    print("Running on CPU (Consider switching runtime to GPU for faster GAN training)")

In [ ]:
# Install dependencies if running directly on Colab
!pip install -q pyyaml opencv-python pillow scikit-learn matplotlib seaborn pandas tqdm scipy

## Step 2: Data Engineering & Source-Level Splitting
We extract fixed-size (128x128) patches. To guarantee **zero data leakage**, splits are made strictly by `source_image_id` before any crops or blur variants are generated.

In [ ]:
import os
import sys

# Ensure project root is in sys.path
sys.path.append(os.getcwd())

from src.data.synthetic_samples import generate_synthetic_road_dataset

# Generate reproducible dataset (or point to parsed RDD2022 patches)
data_dir = "data/processed"
manifest_path, counts_path = generate_synthetic_road_dataset(
    output_dir=data_dir,
    num_images=120,
    seed=42
)

import json
with open(counts_path) as f:
    print("Class Distribution across Splits:")
    print(json.dumps(json.load(f), indent=2))

## Step 3: Train Primary MobileNetV2 Classifier on Clean Patches
We establish the primary reference benchmark: a MobileNetV2 transfer-learning classifier trained solely on sharp road patches.

In [ ]:
from src.train.train_classifier import train_classifier_pipeline

clean_clf_ckpt, clean_history = train_classifier_pipeline(
    config_path="configs/config.yaml",
    architecture="mobilenet_v2",
    use_blur_augmentation=False,
    override_epochs=15
)

## Step 4: Train Conditional GAN (U-Net Generator + PatchGAN Discriminator)
The cGAN is trained on aligned blurry-sharp patch pairs. It combines adversarial loss with L1 reconstruction loss to restore high-frequency crack edges and pothole cavities.

In [ ]:
from src.train.train_gan import train_gan_pipeline

gan_ckpt, gan_history = train_gan_pipeline(
    config_path="configs/config.yaml",
    override_epochs=20
)

## Step 5: (Optional) Train Blur-Augmented Classifier
As a secondary comparison, we evaluate whether data augmentation alone is sufficient to handle road blur without a dedicated restoration GAN.

In [ ]:
blur_clf_ckpt, blur_history = train_classifier_pipeline(
    config_path="configs/config.yaml",
    architecture="mobilenet_v2",
    use_blur_augmentation=True,
    override_epochs=15
)

## Step 6: Controlled Comparative Experiment
We evaluate the **same fixed classifier** on the **exact same held-out test split** across four conditions:
1. **Exp A:** Clean Reference
2. **Exp B:** Synthetic Blurred (Motion & Defocus)
3. **Exp C:** Classical Restoration (Unsharp Masking)
4. **Exp D:** GAN Restoration (Ours)
5. **Exp E:** Blur-Augmented Classifier

In [ ]:
from src.evaluate.comparative_experiment import run_comparative_experiment

results = run_comparative_experiment(
    config_path="configs/config.yaml",
    classifier_clean_ckpt=clean_clf_ckpt,
    gan_ckpt=gan_ckpt,
    classifier_blur_ckpt=blur_clf_ckpt,
    output_dir="results"
)

## Step 7: Visual Quality & Error Analysis

In [ ]:
from IPython.display import Image, display

# Display confusion matrices
for exp in ["Exp_A_Clean", "Exp_B_Blurred", "Exp_C_Classical_Restored", "Exp_D_GAN_Restored"]:
    cm_file = f"results/figures/confusion_matrices/{exp}_cm.png"
    if os.path.exists(cm_file):
        print(f"=== {exp} ===")
        display(Image(filename=cm_file))

# Read and display summary table
with open("results/comparative_summary_table.md") as f:
    print(f.read())